In [14]:
import os
import kagglehub
import pandas as pd
from transformers import AutoTokenizer
import re

# Download the dataset (returns the folder path)
path = kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")

# FIX: Join the folder path with the actual filename
csv_path = os.path.join(path, "IMDB Dataset.csv")

# Now read the specific file
df = pd.read_csv(csv_path)

print("Success!")

Success!


In [15]:
df

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive
...,...,...
49995,I thought this movie did a down right good job...,positive
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,I am a Catholic taught in parochial elementary...,negative
49998,I'm going to have to disagree with the previou...,negative


In [16]:
from datasets import Dataset

In [9]:
dataset.loc[1,:]["review"]

'A wonderful little production. <br /><br />The filming technique is very unassuming- very old-time-BBC fashion and gives a comforting, and sometimes discomforting, sense of realism to the entire piece. <br /><br />The actors are extremely well chosen- Michael Sheen not only "has got all the polari" but he has all the voices down pat too! You can truly see the seamless editing guided by the references to Williams\' diary entries, not only is it well worth the watching but it is a terrificly written and performed piece. A masterful production about one of the great master\'s of comedy and his life. <br /><br />The realism really comes home with the little things: the fantasy of the guard which, rather than use the traditional \'dream\' techniques remains solid then disappears. It plays on our knowledge and our senses, particularly with the scenes concerning Orton and Halliwell and the sets (particularly of their flat with Halliwell\'s murals decorating every surface) are terribly well d

## Work embedding

In [17]:
# Load into a Hugging Face Dataset object
dataset = Dataset.from_csv(csv_path)
# 2. Initialize the Tokenizer
# DistilBERT is a great balance of speed and accuracy
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

# 3. Define the Cleaning & Tokenizing Function
def preprocess_function(examples):
    # 'examples' is a dictionary containing a BATCH of 1000 reviews
    
    # A. Clean HTML tags for the whole batch at once
    # We use a list comprehension here which is very fast
    cleaned_texts = [re.sub(r'<.*?>', ' ', text) for text in examples['review']]
    
    # B. Tokenize the batch
    # truncation=True: Cut reviews longer than 512 tokens (model limit)
    # padding="max_length": Pad shorter reviews with 0s so they are all equal length
    return tokenizer(cleaned_texts, truncation=True, padding="max_length", max_length=512)

# 4. Apply to the whole dataset efficiently
# batched=True is the secret sauce here. It processes 1000 rows at a time.
tokenized_dataset = dataset.map(preprocess_function, batched=True)

# --- Inspect the Result ---
print(tokenized_dataset)
# You will now see new columns: 'input_ids', 'attention_mask'

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

Dataset({
    features: ['review', 'sentiment', 'input_ids', 'attention_mask'],
    num_rows: 50000
})


In [28]:
id0 = tokenized_dataset[0]['input_ids']

In [29]:
model_view = tokenizer.decode(id0)

In [30]:
print(model_view)

[CLS] one of the other reviewers has mentioned that after watching just 1 oz episode you ' ll be hooked. they are right, as this is exactly what happened with me. the first thing that struck me about oz was its brutality and unflinching scenes of violence, which set in right from the word go. trust me, this is not a show for the faint hearted or timid. this show pulls no punches with regards to drugs, sex or violence. its is hardcore, in the classic use of the word. it is called oz as that is the nickname given to the oswald maximum security state penitentary. it focuses mainly on emerald city, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. em city is home to many.. aryans, muslims, gangstas, latinos, christians, italians, irish and more.... so scuffles, death stares, dodgy dealings and shady agreements are never far away. i would say the main appeal of the show is due to the fact that it goes where ot

## discretize labels

In [31]:
# Create a simple mapping function
def map_labels(example):
    return {"label": 1 if example["sentiment"] == "positive" else 0}

# Apply it to the dataset
tokenized_dataset = tokenized_dataset.map(map_labels)

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [34]:
tokenized_dataset.set_format(
    type="torch", 
    columns=["input_ids", "attention_mask", "label"]
)

In [35]:
tokenized_dataset[0]

{'input_ids': tensor([  101,  2028,  1997,  1996,  2060, 15814,  2038,  3855,  2008,  2044,
          3666,  2074,  1015, 11472,  2792,  2017,  1005,  2222,  2022, 13322,
          1012,  2027,  2024,  2157,  1010,  2004,  2023,  2003,  3599,  2054,
          3047,  2007,  2033,  1012,  1996,  2034,  2518,  2008,  4930,  2033,
          2055, 11472,  2001,  2049, 24083,  1998,  4895, 10258,  2378,  8450,
          5019,  1997,  4808,  1010,  2029,  2275,  1999,  2157,  2013,  1996,
          2773,  2175,  1012,  3404,  2033,  1010,  2023,  2003,  2025,  1037,
          2265,  2005,  1996,  8143, 18627,  2030,  5199,  3593,  1012,  2023,
          2265,  8005,  2053, 17957,  2007, 12362,  2000,  5850,  1010,  3348,
          2030,  4808,  1012,  2049,  2003, 13076,  1010,  1999,  1996,  4438,
          2224,  1997,  1996,  2773,  1012,  2009,  2003,  2170, 11472,  2004,
          2008,  2003,  1996,  8367,  2445,  2000,  1996, 17411,  4555,  3036,
          2110,  7279,  4221, 12380,  2

In [36]:
# 80% Train, 20% Test
split_dataset = tokenized_dataset.train_test_split(test_size=0.2, seed=42)

train_data = split_dataset['train']
test_data = split_dataset['test']

In [38]:
from torch.utils.data import DataLoader

# Batch size of 16 or 32 is standard for fine-tuning Transformers
BATCH_SIZE = 16

train_loader = DataLoader(
    train_data, 
    batch_size=BATCH_SIZE, 
    shuffle=True  # Always shuffle training data!
)

test_loader = DataLoader(
    test_data, 
    batch_size=BATCH_SIZE, 
    shuffle=False # No need to shuffle validation data
)

In [39]:
batch0 = next(iter(train_loader))

In [40]:
batch0

{'input_ids': tensor([[ 101, 2023, 3925,  ...,    0,    0,    0],
         [ 101, 2129, 3087,  ...,    0,    0,    0],
         [ 101, 2043, 2111,  ..., 2008, 2009,  102],
         ...,
         [ 101, 1045, 2387,  ..., 2205, 1024,  102],
         [ 101, 2045, 2003,  ...,    0,    0,    0],
         [ 101, 1042, 4140,  ...,    0,    0,    0]]),
 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 1, 1, 1],
         ...,
         [1, 1, 1,  ..., 1, 1, 1],
         [1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 0, 0, 0]]),
 'label': tensor([0, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0])}

## Transformer-based classifer

In [ ]:
class TransformerClassifier:
    def __init__():
        pass

    
    def forward():
        pass